<img src="https://sfile.chatglm.cn/images-ppt/cfe256b11254.jpg" width="500" align="center">

# От блокнота к сервису — обучаем модель и упаковываем в API

**Роль:** Преподаватель по ML  
**Уровень:** средний (основы Python + базовое понимание ML)  
**Время прохождения:** ~60–90 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- обучите реальную модель классификации изображений (кошки vs собаки);
- **своими руками** сохраните модель и загрузите обратно;
- поймёте, чем «обучение в блокноте» отличается от «модель в продакшене»;
- **сами** напишете FastAPI-сервис, принимающий картинку и возвращающий прогноз;
- протестируете сервис прямо из блокнота;
- увидите типичные ошибки и научитесь их читать.

### Принцип этого блокнота

> Вы — автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких «чёрных ящиков».

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Знакомство с данными | Скачиваем датасет, смотрим на картинки |
| 3 | Обучаем модель | Transfer learning с resnet18 |
| 4 | Что внутри модели? | Разбираем архитектуру и предсказания |
| 5 | Интерактивный тестер | Виджет для проверки модели на своих картинках |
| 6 | Сохраняем и загружаем | Два способа, плюсы и минусы |
| 7 | Блокнот vs продакшен | Ключевые различия |
| 8 | Пишем FastAPI-сервис | Каждый файл, каждая строка |
| 9 | Запускаем и тестируем | 6 тестов, включая «плохие» запросы |
| 10 | Типичные ошибки | Ломаем код и учимся читать ошибки |
| 11 | Практические задания | 5 задач для самостоятельной работы |
| 12 | Дальше | Куда двигаться |

---
## Шаг 1. Подготовка окружения

Нам понадобятся:

| Библиотека | Зачем |
|-----------|-------|
| **fastai** | Обучение модели на реальных данных в несколько строк (поверх PyTorch) |
| **fastapi** + **uvicorn** | Создание HTTP-API и сервер для его запуска |
| **ipywidgets** | Интерактивные виджеты прямо в блокноте |
| **Pillow** | Работа с изображениями |
| **httpx** | HTTP-клиент для тестирования API |

Почему fastai, а не «чистый» PyTorch? В предыдущих блокнотах мы уже обучали модели вручную — это важно для понимания. Сейчас наша цель другая: **пройти весь путь от обучения до рабочего сервиса**. fastai экономит время на рутине, но мы всё равно будем заглядывать внутрь.

In [ ]:
# Установка всех зависимостей (в Colab fastai уже есть, но перестрахуемся)
!pip install -q fastai fastapi uvicorn python-multipart Pillow httpx ipywidgets

# Включаем интерактивные виджеты в Colab
!jupyter nbextension enable --py widgetsnbextension 2>/dev/null || true

import sys
print(f'Python:  {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')

import fastai
print(f'fastai:  {fastai.__version__}')

import fastapi
print(f'fastapi: {fastapi.__version__}')

import ipywidgets as widgets
print(f'ipywidgets: OK')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nУстройство: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('GPU недоступен — обучение будет на CPU (медленнее, но работает)')

print('\n✅ Окружение готово!')

---
## Шаг 2. Знакомство с данными

Прежде чем обучать модель, нужно **понять данные**. Это правило номер один в ML: сначала смотри на данные, потом делай всё остальное.

Мы используем **Oxford-IIIT Pet Dataset** — 7 349 фотографий домашних животных 37 пород. Мы сведём задачу к бинарной: кошка vs собака.

In [ ]:
from fastai.vision.all import *

# Скачиваем датасет (~750 МБ)
path = untar_data(URLs.PETS)
print(f'Датасет: {path}')
print(f'Папки: {path.ls()}')

# Картинки лежат в images/
img_path = path / 'images'
image_files = get_image_files(img_path)
print(f'\nВсего изображений: {len(image_files)}')

In [ ]:
# Как устроены имена файлов?
# Кошки: первая буква ЗАГЛАВНАЯ — Abyssinian_1.jpg, Persian_23.jpg
# Собаки: первая буква строчная — beagle_5.jpg, pug_100.jpg

print('Примеры файлов:')
for f in sorted(image_files)[:10]:
    animal = '🐱 КОШКА' if f.name[0].isupper() else '🐶 СОБАКА'
    print(f'  {f.name:30s} -> {animal}')

print(f'\n...и так далее, всего {len(image_files)} файлов')

# Функция-метка
def is_cat(filename):
    """True = кошка, False = собака."""
    return filename[0].isupper()

# Считаем баланс классов
cats = sum(1 for f in image_files if is_cat(f.name))
dogs = len(image_files) - cats
print(f'\nКошек: {cats} ({cats/len(image_files):.1%})')
print(f'Собак: {dogs} ({dogs/len(image_files):.1%})')

In [ ]:
# Интерактивный просмотр картинок!
# Нажимайте кнопки, чтобы листать датасет

import ipywidgets as widgets
from IPython.display import display, clear_output

idx_slider = widgets.IntSlider(
    value=0, min=0, max=len(image_files)-1,
    description='Индекс:',
    layout=widgets.Layout(width='500px')
)

output = widgets.Output()

def show_image(idx):
    with output:
        clear_output(wait=True)
        f = image_files[idx]
        img = PILImage.create(f)
        label = '🐱 КОШКА' if is_cat(f.name) else '🐶 СОБАКА'
        
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'{label}\n{f.name}\nРазмер: {img.size}', fontsize=12)
        plt.tight_layout()
        plt.show()

widgets.interact(show_image, idx=idx_slider)
display(output)

### Что вы заметили?

Пролистайте 20–30 картинок и обратите внимание:
- Разные размеры — модель требует одинаковый размер входа.
- Разный фон, освещение, ракурс — модель должна быть устойчива к этому.
- Иногда животное не в фокусе или частично скрыто — это нормально для реальных данных.

Всё это модель будет «видеть» и учиться обрабатывать. Именно поэтому нужно много данных — чтобы модель увидела достаточно вариаций.

---
## Шаг 3. Обучаем модель

<img src="https://sfile.chatglm.cn/images-ppt/1eabcf97adfd.png" width="600" align="center">

Мы используем **transfer learning** (трансферное обучение): берём модель, уже обученную на 1.2 млн изображений ImageNet, и дообучаем на нашу задачу.

Аналогия: вы наняли шеф-повара с 10-летним стажем итальянской кухни. Вам не нужно учить его резать лук — нужно только показать, какие специи вы предпочитаете. Модель уже умеет видеть грани, текстуры, формы — мы учим её, какие из них означают «кошка», а какие «собака».

In [ ]:
# Создаём DataLoaders — конвейер данных для модели
# Разбираем КАЖДЫЙ параметр:

dls = ImageDataLoaders.from_name_func(
    path,                              # корневая папка
    get_image_files(img_path),         # список файлов
    valid_pct=0.2,                     # 20% данных -> валидация (модель их НЕ видит при обучении!)
    seed=42,                           # фиксируем разбиение (воспроизводимость)
    label_func=is_cat,                 # функция метки: кошка или собака
    item_tfms=Resize(224),             # масштабируем до 224x224 (стандартный размер для resnet)
    # попробуйте: Resize(128) — быстрее, но хуже качество
    # или:          Resize(384) — медленнее, но точнее
)

print(f'Обучающая выборка:   {len(dls.train.ds):,} изображений')
print(f'Валидационная выборка: {len(dls.valid.ds):,} изображений')
print(f'Классы: {dls.vocab}')  # [False, True] = [собака, кошка]

In [ ]:
# Смотрим, что видит модель — батч тренировочных данных
dls.show_batch(max_n=8, nrows=2, figsize=(14, 6))
print('Обратите внимание: все картинки приведены к размеру 224x224.')
print('Это нужно, потому что нейросеть ожидает фиксированный размер входа.')

In [ ]:
# Создаём learner — объект, который управляет обучением

# resnet18 — свёрточная сеть из 18 слоёв, предобученная на ImageNet
# попробуйте: resnet34 (точнее, но в 2x медленнее) или resnet50 (ещё точнее, ещё медленнее)
learn = vision_learner(dls, resnet18, metrics=accuracy)

print(f'Архитектура: resnet18')
print(f'Всего параметров: {sum(p.numel() for p in learn.model.parameters()):,}')
print(f'Из них обучаемых: {sum(p.numel() for p in learn.model.parameters() if p.requires_grad):,}')
print()
print('Параметры — это «ручки», которые модель крутит в процессе обучения.')
print('11+ миллионов ручек — и обучение находит правильное положение для каждой!')

In [ ]:
# Обучаем!

# fine_tune(3) делает следующее:
# Эпоха 1: замораживает ВСЕ слои кроме последнего, обучает только его (быстро)
# Эпохи 2-3: размораживает все слои, обучает с маленьким learning rate (аккуратно)
#
# попробуйте изменить: 2 (мало), 5 (нормально), 10 (много)

print('Начинаем обучение... (это займёт 2-5 минут)')
learn.fine_tune(3)

In [ ]:
# Визуализируем результаты обучения

learn.recorder.plot_loss(figsize=(8, 4))
plt.title('Loss по эпохам')
plt.grid(True, alpha=0.3)
plt.show()

# Точность на валидации
val_acc = learn.validate()[1]
print(f'Точность на валидации: {val_acc:.2%}')
print()
if val_acc > 0.98:
    print('🔥 Отлично! Модель почти не ошибается.')
elif val_acc > 0.95:
    print('✅ Хорошо! Модель работает нормально.')
elif val_acc > 0.90:
    print('⚠️ Можно лучше. Попробуйте больше эпох или resnet34.')
else:
    print('❌ Модель недообучена. Проверьте данные и параметры.')

In [ ]:
# Смотрим, где модель ошибается

interp = ClassificationInterpretation.from_learner(learn)

# Матрица ошибок (confusion matrix)
interp.plot_confusion_matrix(figsize=(4, 4))
plt.show()

# Топ-9 самых «сложных» картинок — где модель меньше всего уверена
interp.plot_top_losses(9, nrows=3, figsize=(14, 8))
plt.show()

print('Картинки сверху — самые трудные для модели.')
print('Обратите внимание: многие из них действительно неоднозначные!')

---
## Шаг 4. Что внутри модели?

Давайте разберём, что именно возвращает `predict()` — шаг за шагом, без «магии».

In [ ]:
# Разбираем predict() по шагам

# Берём тестовую картинку
test_file = image_files[42]
test_img = PILImage.create(test_file)

print(f'Файл: {test_file.name}')
print(f'Размер: {test_img.size}')
print(f'Тип: {type(test_img)}')

# ШАГ 1: predict() делает препроцессинг
# - Ресайзит до 224x224
# - Нормализует пиксели (вычитает среднее, делит на std ImageNet)
# Это АВТОМАТИЧЕСКИ — fastai запоминает трансформации из dls

# ШАГ 2: прямой проход через нейросеть
# - Вход: (1, 3, 224, 224) тензор (батч=1, 3 канала RGB, 224x224)
# - 18 слоёв свёрток + активаций
# - Выход: 2 числа (логиты) — «сырая» оценка для каждого класса

# ШАГ 3: softmax превращает логиты в вероятности
# - Сумма вероятностей = 1.0

pred_class, pred_idx, probs = learn.predict(test_img)

print(f'\n=== Результат ===')
print(f'Предсказанный класс: {pred_class}  (True=кошка, False=собака)')
print(f'Индекс класса: {pred_idx}')
print(f'Вероятности: собака={probs[0]:.4f}, кошка={probs[1]:.4f}')
print(f'Уверенность: {max(probs):.2%}')
print(f'\nЭто {"🐱 КОШКА" if pred_class else "🐶 СОБАКА"} с уверенностью {max(probs):.1%}')

In [ ]:
# ВАЖНЫЙ ЭКСПЕРИМЕНТ: что если подать «мусор»?

# Создаём случайный шум
random_noise = torch.randint(0, 256, (224, 224, 3), dtype=torch.uint8)
random_pil = PILImage.create(random_noise.numpy())

pred_class, pred_idx, probs = learn.predict(random_pil)

print('Подали случайный шум (не кошку и не собаку):')
print(f'Модель ответила: {"🐱 КОШКА" if pred_class else "🐶 СОБАКА"} с уверенностью {max(probs):.2%}')
print()
print('⚠️ МОДЕЛЬ ВСЕГДА ЧТО-ТО ОТВЕТИТ! Она не умеет говорить «я не знаю».')
print('Это фундаментальная проблема, которую нужно решать в продакшене.')
print('Решение: добавить порог уверенности (сделаем это позже).')

---
## Шаг 5. Интерактивный тестер модели

Прежде чем упаковывать модель в сервис, давайте поиграемся с ней прямо в блокноте.

Ниже — виджет, который позволяет вам:
- листать картинки из датасета и видеть предсказания;
- загрузить свою картинку (если запускаете локально).

In [ ]:
# Интерактивный тестер модели — листаем датасет

idx_slider = widgets.IntSlider(
    value=0, min=0, max=len(image_files)-1, step=1,
    description='Картинка #:',
    layout=widgets.Layout(width='500px')
)

threshold_slider = widgets.FloatSlider(
    value=0.7, min=0.5, max=0.99, step=0.01,
    description='Порог:',
    layout=widgets.Layout(width='300px')
)

output = widgets.Output()

def test_model(idx, threshold):
    with output:
        clear_output(wait=True)
        f = image_files[idx]
        img = PILImage.create(f)
        true_label = '🐱 КОШКА' if is_cat(f.name) else '🐶 СОБАКА'
        
        pred_class, _, probs = learn.predict(img)
        confidence = float(max(probs))
        pred_label = '🐱 КОШКА' if pred_class else '🐶 СОБАКА'
        reliable = confidence >= threshold
        
        # Рисуем
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(img)
        ax.axis('off')
        
        if reliable:
            color = 'green' if pred_label == true_label else 'red'
            status = '✅ Верно' if pred_label == true_label else '❌ ОШИБКА'
            ax.set_title(f'{pred_label} ({confidence:.1%})\n{status}', 
                        fontsize=14, color=color, fontweight='bold')
        else:
            ax.set_title(f'⚠️ НЕ УВЕРЕН\n{pred_label} ({confidence:.1%})', 
                        fontsize=14, color='orange', fontweight='bold')
        
        plt.tight_layout()
        plt.show()
        
        print(f'Файл: {f.name}')
        print(f'Истинный класс: {true_label}')
        print(f'Предсказание:   {pred_label}')
        print(f'Уверенность:    {confidence:.2%}')
        print(f'Порог:          {threshold:.0%}')
        print(f'Надёжно:        {"Да" if reliable else "Нет"}')
        print(f'Вероятности:    кошка={probs[1]:.4f}, собака={probs[0]:.4f}')

widgets.interact(test_model, idx=idx_slider, threshold=threshold_slider)
display(output)

### Поиграйте с порогом!

- Установите порог **0.50** — модель почти всегда «уверена», но иногда ошибается.
- Установите порог **0.95** — модель часто «не уверена», но среди уверенных ответов почти нет ошибок.

Это классический **trade-off** (компромисс) в ML: чем строже порог, тем меньше ошибок, но и больше отказов от ответа. В продакшене вы выбираете порог исходя из задачи: для медицинского диагноза нужен высокий порог, для рекомендаций фильмов — низкий.

---
## Шаг 6. Сохраняем и загружаем модель

Обученную модель нужно сохранить в файл. Зачем?
- Не обучать каждый раз заново (обучение занимает минуты/часы)
- Перенести на сервер, где будет работать API
- Поделиться с коллегами

Есть два способа. Давайте попробуем оба и сравним.

In [ ]:
# Способ 1: learn.export() — рекомендуется для продакшена

# Сохраняет ВСЁ: архитектуру + веса + препроцессинг + словарь классов
learn.export('cat_dog_model.pkl')

import os
model_path = 'cat_dog_model.pkl'
size_mb = os.path.getsize(model_path) / 1e6
print(f'✅ Модель сохранена: {model_path} ({size_mb:.1f} МБ)')

# Загружаем обратно
loaded_learn = load_learner(model_path)

# Проверяем: совпадают ли результаты?
test_img = PILImage.create(image_files[200])
p1, _, pr1 = learn.predict(test_img)
p2, _, pr2 = loaded_learn.predict(test_img)

print(f'\nОригинальная модель:  {"кошка" if p1 else "собака"} ({max(pr1):.4f})')
print(f'Загруженная модель:   {"кошка" if p2 else "собака"} ({max(pr2):.4f})')
print(f'Результаты совпадают: {p1 == p2 and abs(float(max(pr1)) - float(max(pr2))) < 1e-6}')

In [ ]:
# Способ 2: torch.save(state_dict) — низкоуровневый

# Сохраняет ТОЛЬКО веса (без архитектуры, без препроцессинга)
torch.save(learn.model.state_dict(), 'cat_dog_weights.pth')

weights_mb = os.path.getsize('cat_dog_weights.pth') / 1e6
print(f'✅ Веса сохранены: cat_dog_weights.pth ({weights_mb:.1f} МБ)')
print(f'Разница в размере: {size_mb - weights_mb:.1f} МБ (в .pkl хранится ещё архитектура + препроцессинг)')

# Чтобы загрузить через state_dict, нужно:
# 1. Создать модель с ТАКОЙ ЖЕ архитектурой
# 2. Загрузить веса
# 3. Самому делать препроцессинг (ресайз, нормализация)
print('\n⚠️ При загрузке через state_dict нужно вручную:')
print('   1. Создать модель с той же архитектурой')
print('   2. Загрузить веса')
print('   3. Делать препроцессинг вручную (ресайз, нормализация)')
print('\nИменно поэтому learn.export() удобнее для продакшена.')

### Сравнение способов

| | `learn.export()` | `torch.save(state_dict)` |
|---|---|---|
| **Что сохраняет** | Всё (архитектура + веса + препроцессинг) | Только веса |
| **При загрузке** | `load_learner()` — сразу работает | Нужна архитектура + препроцессинг руками |
| **Размер** | Больше (~47 МБ) | Меньше (~45 МБ) |
| **Зависимость** | Нужен fastai | Нужен только PyTorch |
| **Безопасность** | pickle (потенциально опасно) | Безопасно |

**Правило безопасности:** никогда не загружайте `.pkl` файлы от незнакомых людей — pickle может выполнить произвольный код при загрузке. Используйте только модели, которые обучили сами или получили из доверенного источника.

---
## Шаг 7. Блокнот vs продакшен

Это **самая важная часть** ноутбука. Прежде чем писать код сервиса, нужно понять, чем он отличается от блокнота.

In [ ]:
# Визуализация: путь от блокнота к продакшену

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Блокнот
axes[0].set_xlim(0, 10)
axes[0].set_ylim(0, 10)
axes[0].axis('off')

boxes_notebook = [
    (1, 8, 'Ваши данные\n(контролируете вы)', 'lightblue'),
    (1, 6, 'Модель в памяти\n(пока открыт ноутбук)', 'lightyellow'),
    (1, 4, 'Результат\n(в ячейке ниже)', 'lightgreen'),
    (1, 2, 'ОШИБКА = стоп\n(чините вручную)', 'lightsalmon'),
]

axes[0].set_title('📓 Блокнот', fontsize=16, fontweight='bold', pad=15)
for x, y, text, color in boxes_notebook:
    axes[0].add_patch(plt.Rectangle((x, y-0.7), 8, 1.4, fc=color, ec='gray', lw=2, alpha=0.8))
    axes[0].text(x+4, y, text, ha='center', va='center', fontsize=11)

for y_start, y_end in [(8-0.7, 6+0.7), (6-0.7, 4+0.7), (4-0.7, 2+0.7)]:
    axes[0].annotate('', xy=(5, y_end), xytext=(5, y_start),
                     arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

axes[0].text(5, 0.5, 'Один пользователь — вы', ha='center', fontsize=11, 
             fontstyle='italic', color='gray')

# Продакшен
axes[1].set_xlim(0, 10)
axes[1].set_ylim(0, 10)
axes[1].axis('off')

boxes_prod = [
    (1, 8, 'Любые данные\n(пользователь шлёт что угодно)', 'lightblue'),
    (1, 6, 'Модель из файла\n(работает 24/7)', 'lightyellow'),
    (1, 4, 'JSON-ответ\n(понятный клиенту)', 'lightgreen'),
    (1, 2, 'ОШИБКА = понятное сообщение\n(+ лог для разработчика)', 'lightsalmon'),
]

axes[1].set_title('🚀 Продакшен', fontsize=16, fontweight='bold', pad=15)
for x, y, text, color in boxes_prod:
    axes[1].add_patch(plt.Rectangle((x, y-0.7), 8, 1.4, fc=color, ec='gray', lw=2, alpha=0.8))
    axes[1].text(x+4, y, text, ha='center', va='center', fontsize=11)

for y_start, y_end in [(8-0.7, 6+0.7), (6-0.7, 4+0.7), (4-0.7, 2+0.7)]:
    axes[1].annotate('', xy=(5, y_end), xytext=(5, y_start),
                     arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

axes[1].text(5, 0.5, 'Много пользователей одновременно', ha='center', fontsize=11, 
             fontstyle='italic', color='gray')

plt.suptitle('Блокнот vs Продакшен: в чём разница?', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Таблица различий — интерактивный виджет

aspects = widgets.Accordion(children=[
    widgets.HTML('<b>В блокноте:</b> Вы контролируете входные данные<br><b>В продакшене:</b> Пользователь может отправить что угодно — PDF, пустой файл, 100 МБ картинку. Нужна валидация!'),
    widgets.HTML('<b>В блокноте:</b> Модель живёт, пока работает ядро<br><b>В продакшене:</b> Модель загружается из файла при старте сервера. Сервер работает 24/7.'),
    widgets.HTML('<b>В блокноте:</b> Результат видите вы в ячейке<br><b>В продакшене:</b> Результат отдаётся как JSON через HTTP. Клиент — другая программа, не человек.'),
    widgets.HTML('<b>В блокноте:</b> Ошибка = стоп, чините вручную<br><b>В продакшене:</b> Ошибка = понятное сообщение клиенту + лог для разработчика. Сервер НЕ падает.'),
    widgets.HTML('<b>В блокноте:</b> Один пользователь — вы<br><b>В продакшене:</b> Много пользователей одновременно. Нужна конкурентность.'),
    widgets.HTML('<b>В блокноте:</b> Модель всегда «уверена»<br><b>В продакшене:</b> Нужен порог уверенности. Лучше сказать «не знаю», чем ошибиться.'),
])

aspects.set_title(0, '🔒 Входные данные')
aspects.set_title(1, '💾 Хранение модели')
aspects.set_title(2, '📤 Формат ответа')
aspects.set_title(3, '⚠️ Обработка ошибок')
aspects.set_title(4, '👥 Пользователи')
aspects.set_title(5, '🎯 Уверенность модели')

display(aspects)
print('\n👆 Нажмите на каждый раздел, чтобы раскрыть подробности')

---
## Шаг 8. Пишем FastAPI-сервис

<img src="https://sfile.chatglm.cn/images-ppt/790d390f0534.jpg" width="400" align="center">

Теперь мы напишем сервис — **прямо в блокноте**, строку за строкой. Вы видите каждый файл, каждую строку.

Архитектура:
```
app/
├── main.py              # API-эндпоинты (HTTP-слой)
├── model_service.py     # Логика модели (ML-слой)
├── cat_dog_model.pkl    # Файл обученной модели
└── requirements.txt     # Зависимости
```

Почему два файла, а не один? **Разделение ответственности** (Separation of Concerns):
- `model_service.py` знает только про модель (загрузка, предсказание)
- `main.py` знает только про HTTP (запросы, ответы, маршруты)

Это позволяет тестировать логику модели без запуска сервера, менять модель не трогая API, и переиспользовать model_service в других проектах.

In [ ]:
# Создаём директорию и копируем модель

import os, shutil
os.makedirs('app', exist_ok=True)
shutil.copy('cat_dog_model.pkl', 'app/cat_dog_model.pkl')
print('✅ app/ создана, модель скопирована')

In [ ]:
# Файл 1: model_service.py — логика модели
# Читайте каждую строку — комментарий объясняет, зачем она нужна

model_service_code = '''"""
model_service.py — инкапсулирует логику загрузки модели и предсказаний.

Зачем отдельный файл? Принцип разделения ответственности:
- Этот файл знает только про модель (загрузка, предсказание)
- main.py знает только про HTTP (запросы, ответы, маршруты)
"""

import io
import torch
from fastai.vision.all import load_learner, PILImage
from PIL import Image
from pathlib import Path


# ========== КОНСТАНТЫ ==========
# Все настраиваемые параметры — в одном месте, чтобы легко менять
MODEL_PATH = Path(__file__).parent / "cat_dog_model.pkl"
CONFIDENCE_THRESHOLD = 0.7  # ниже этого порога -> "не уверен"
# попробуйте: 0.5 (мягче), 0.9 (строже)

MAX_IMAGE_SIZE = 10 * 1024 * 1024  # 10 МБ — максимальный размер файла
ALLOWED_TYPES = ["image/jpeg", "image/png", "image/webp"]  # допустимые форматы


# ========== ЗАГРУЗКА МОДЕЛИ ==========
# Глобальная переменная: модель загружается ОДИН РАЗ при старте сервера,
# а не при каждом запросе! Загрузка занимает секунды.
_model = None

def load_model():
    """Загружает модель из файла. Вызывается один раз при старте."""
    global _model
    if _model is None:
        print(f"Загрузка модели из {MODEL_PATH}...")
        _model = load_learner(MODEL_PATH)
        print("Модель загружена!")
    return _model


# ========== ПРЕДСКАЗАНИЕ ==========
def predict(image_bytes: bytes) -> dict:
    """
    Принимает байты изображения, возвращает словарь с результатом.
    
    Шаги:
    1. Конвертируем байты -> PIL-изображение
    2. Вызываем model.predict()
    3. Проверяем уверенность
    4. Формируем ответ
    """
    model = load_model()
    
    # Шаг 1: байты -> изображение
    # try/except потому что пользователь может прислать НЕ картинку
    try:
        img = Image.open(io.BytesIO(image_bytes))
        img = PILImage.create(img)
    except Exception as e:
        return {"error": f"Не удалось прочитать изображение: {str(e)}"}
    
    # Шаг 2: предсказание
    try:
        pred_class, pred_idx, probs = model.predict(img)
    except Exception as e:
        return {"error": f"Ошибка модели: {str(e)}"}
    
    # Шаг 3: проверяем уверенность
    confidence = float(max(probs))
    is_reliable = confidence >= CONFIDENCE_THRESHOLD
    
    # Шаг 4: формируем ответ
    label = "кот" if pred_class else "собака"
    
    return {
        "class": label if is_reliable else "не уверен",
        "confidence": round(confidence, 4),
        "is_reliable": is_reliable,
        "prob_cat": round(float(probs[1]), 4),
        "prob_dog": round(float(probs[0]), 4),
    }
'''

with open('app/model_service.py', 'w', encoding='utf-8') as f:
    f.write(model_service_code.strip())

print('✅ app/model_service.py создан')
print(f'   Размер: {os.path.getsize("app/model_service.py")} байт')

In [ ]:
# Файл 2: main.py — API-сервер

main_code = '''"""
main.py — FastAPI-сервер для классификации кошек/собак.

Эндпоинты:
- GET  /         -> health check (сервер работает?)
- GET  /info     -> информация о модели
- POST /predict  -> классификация изображения
"""

from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
import time

from model_service import load_model, predict, MODEL_PATH, CONFIDENCE_THRESHOLD, MAX_IMAGE_SIZE, ALLOWED_TYPES


# Создаём приложение
app = FastAPI(
    title="Cat/Dog Classifier API",
    description="API для классификации: кошка или собака",
    version="1.0.0",
)


# ========== STARTUP ==========
# Выполняется один раз при запуске сервера
@app.on_event("startup")
async def startup_event():
    """Загружаем модель при старте, а не при первом запросе."""
    load_model()
    print("Сервер готов к работе!")


# ========== HEALTH CHECK ==========
@app.get("/")
async def root():
    """Проверка: сервер работает?"""
    return {"status": "ok", "service": "Cat/Dog Classifier", "model_loaded": True}


# ========== ИНФОРМАЦИЯ О МОДЕЛИ ==========
@app.get("/info")
async def info():
    """Информация о модели и настройках."""
    model = load_model()
    return {
        "model_path": str(MODEL_PATH),
        "classes": {"False": "собака", "True": "кошка"},
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "max_image_size_mb": MAX_IMAGE_SIZE / 1e6,
        "allowed_types": ALLOWED_TYPES,
    }


# ========== КЛАССИФИКАЦИЯ ==========
@app.post("/predict")
async def predict_endpoint(file: UploadFile = File(...)):
    """
    Принимает изображение и возвращает классификацию.
    
    Пример curl:
    curl -X POST -F "file=@photo.jpg" http://localhost:8000/predict
    """
    start_time = time.time()
    
    # ВАЛИДАЦИЯ 1: тип файла
    if file.content_type not in ALLOWED_TYPES:
        raise HTTPException(
            status_code=400,
            detail=f"Неподдерживаемый формат: {file.content_type}. "
                   f"Допускаются: {', '.join(ALLOWED_TYPES)}"
        )
    
    # Читаем файл
    image_bytes = await file.read()
    
    # ВАЛИДАЦИЯ 2: размер файла
    if len(image_bytes) > MAX_IMAGE_SIZE:
        raise HTTPException(
            status_code=400,
            detail=f"Файл слишком большой: {len(image_bytes)/1e6:.1f} МБ. Максимум: {MAX_IMAGE_SIZE/1e6:.0f} МБ"
        )
    
    # Вызываем модель
    result = predict(image_bytes)
    
    # Если predict() вернул ошибку
    if "error" in result:
        raise HTTPException(status_code=422, detail=result["error"])
    
    # Добавляем метаданные
    result["filename"] = file.filename
    result["processing_time_ms"] = round((time.time() - start_time) * 1000, 1)
    
    return result
'''

with open('app/main.py', 'w', encoding='utf-8') as f:
    f.write(main_code.strip())

print('✅ app/main.py создан')
print(f'   Размер: {os.path.getsize("app/main.py")} байт')

In [ ]:
# Файл 3: requirements.txt

requirements = '''fastai>=2.7
fastapi>=0.100
uvicorn>=0.23
python-multipart>=0.0.6
Pillow>=9.0
torch>=1.13
torchvision>=0.14
'''

with open('app/requirements.txt', 'w') as f:
    f.write(requirements.strip())

print('✅ app/requirements.txt создан')
print('\nСтруктура проекта:')
for root, dirs, files in os.walk('app'):
    level = root.replace('app', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        size = os.path.getsize(os.path.join(root, file))
        size_str = f'{size/1e6:.1f} МБ' if size > 1e6 else f'{size} Б'
        print(f'{subindent}{file} ({size_str})')

In [ ]:
# Визуализация архитектуры сервиса

fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 14)
ax.set_ylim(0, 6)
ax.axis('off')

# Пользователь
ax.add_patch(plt.Rectangle((0.3, 2.2), 2, 1.6, fc='lightblue', ec='steelblue', lw=2, alpha=0.8))
ax.text(1.3, 3, 'Пользователь\n(клиент)', ha='center', va='center', fontsize=11, fontweight='bold')

# Стрелка: запрос
ax.annotate('', xy=(3.5, 3.3), xytext=(2.3, 3.3),
            arrowprops=dict(arrowstyle='->', lw=2, color='green'))
ax.text(2.9, 3.6, 'POST /predict\n(файл.jpg)', ha='center', fontsize=8, color='green')

# Стрелка: ответ
ax.annotate('', xy=(2.3, 2.7), xytext=(3.5, 2.7),
            arrowprops=dict(arrowstyle='->', lw=2, color='red'))
ax.text(2.9, 2.1, 'JSON ответ\n{class: "кот"}', ha='center', fontsize=8, color='red')

# FastAPI
ax.add_patch(plt.Rectangle((3.5, 1.5), 2.5, 3, fc='lightyellow', ec='orange', lw=2, alpha=0.8))
ax.text(4.75, 3.3, 'main.py', ha='center', va='center', fontsize=12, fontweight='bold', color='orange')
ax.text(4.75, 2.7, 'FastAPI', ha='center', va='center', fontsize=10, color='gray')
ax.text(4.75, 2.2, 'Валидация\nHTTP-коды', ha='center', va='center', fontsize=9)

# Стрелка
ax.annotate('', xy=(7, 3), xytext=(6, 3),
            arrowprops=dict(arrowstyle='<->', lw=2, color='gray'))

# Model service
ax.add_patch(plt.Rectangle((7, 1.5), 2.5, 3, fc='lightgreen', ec='green', lw=2, alpha=0.8))
ax.text(8.25, 3.3, 'model_service.py', ha='center', va='center', fontsize=11, fontweight='bold', color='green')
ax.text(8.25, 2.5, 'load_model()\npredict()\nпорог уверенности', ha='center', va='center', fontsize=9)

# Стрелка
ax.annotate('', xy=(10.5, 3), xytext=(9.5, 3),
            arrowprops=dict(arrowstyle='<->', lw=2, color='gray'))

# Модель
ax.add_patch(plt.Rectangle((10.5, 1.5), 2.5, 3, fc='lightsalmon', ec='firebrick', lw=2, alpha=0.8))
ax.text(11.75, 3.3, 'cat_dog_model.pkl', ha='center', va='center', fontsize=10, fontweight='bold', color='firebrick')
ax.text(11.75, 2.3, 'resnet18\nкошка/собака\n47 МБ', ha='center', va='center', fontsize=9)

ax.set_title('Архитектура Cat/Dog Classifier API', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## Шаг 9. Запускаем и тестируем сервис

<img src="https://sfile.chatglm.cn/images-ppt/fecac9c1b165.png" width="500" align="center">

Сначала тестируем model_service напрямую (без сервера), потом запустим сервер и отправим HTTP-запросы.

In [ ]:
# Тест 0: model_service без сервера (юнит-тест)

import sys
sys.path.insert(0, 'app')

from model_service import load_model, predict

# Загружаем модель
model = load_model()
print(f'Модель: {type(model)}')
print(f'Классы: {model.dls.vocab}')

# Тест: реальная картинка
with open(image_files[50], 'rb') as f:
    result = predict(f.read())
print(f'\n✅ Реальная картинка: {result}')

# Тест: «мусор» вместо картинки
bad_result = predict(b'this is not an image at all')
print(f'✅ Мусор на входе: {bad_result}')
print('\nМодель НЕ упала! Вернула понятную ошибку.')

In [ ]:
# Запускаем сервер!

import subprocess, time

# Убиваем предыдущий сервер (если был)
!pkill -f 'uvicorn.*main:app' 2>/dev/null || true
time.sleep(1)

# Запускаем в фоне
server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='app', stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

print('Запуск сервера...')
time.sleep(5)  # даём время загрузить модель

# Проверяем
import httpx
try:
    r = httpx.get('http://localhost:8000/', timeout=5)
    print(f'✅ Сервер работает! Статус: {r.status_code}')
    print(f'   Ответ: {r.json()}')
except Exception as e:
    print(f'❌ Не удалось подключиться: {e}')
    print('Подождите 5 секунд и запустите следующую ячейку.')

In [ ]:
# Тест 1: Health check

r = httpx.get('http://localhost:8000/')
print(f'GET / -> {r.status_code}')
print(f'Ответ: {r.json()}')

In [ ]:
# Тест 2: Информация о модели

r = httpx.get('http://localhost:8000/info')
print(f'GET /info -> {r.status_code}')
import json
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

In [ ]:
# Тест 3: Классификация реальной картинки

test_file = image_files[150]

with open(test_file, 'rb') as f:
    r = httpx.post('http://localhost:8000/predict',
                    files={'file': (test_file.name, f, 'image/jpeg')},
                    timeout=30)

result = r.json()
print(f'POST /predict -> {r.status_code}')
print(json.dumps(result, indent=2, ensure_ascii=False))

# Показываем картинку + результат
img = PILImage.create(test_file)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(img)
ax.axis('off')
emoji = '🐱' if result.get('class') == 'кот' else '🐶'
ax.set_title(f'{emoji} {result.get("class", "?")} ({result.get("confidence", 0):.1%})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Тест 4: Серия картинок — визуальная сетка результатов

import random
random.seed(42)
test_indices = random.sample(range(len(image_files)), 9)

fig, axes = plt.subplots(3, 3, figsize=(14, 12))

for idx, ax in zip(test_indices, axes.flat):
    f = image_files[idx]
    with open(f, 'rb') as fh:
        r = httpx.post('http://localhost:8000/predict',
                        files={'file': (f.name, fh, 'image/jpeg')}, timeout=30)
    result = r.json()
    
    img = PILImage.create(f)
    ax.imshow(img)
    ax.axis('off')
    
    label = result.get('class', '?')
    conf = result.get('confidence', 0)
    reliable = result.get('is_reliable', False)
    
    emoji = '🐱' if 'кот' in label else '🐶' if 'собак' in label else '❓'
    color = 'green' if reliable else 'orange'
    ax.set_title(f'{emoji} {label} ({conf:.1%})', fontsize=11, color=color)

plt.suptitle('Результаты классификации через API', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Тест 5: «Плохой» запрос — отправляем текст вместо картинки

r = httpx.post('http://localhost:8000/predict',
               files={'file': ('test.txt', b'Just some text', 'text/plain')},
               timeout=10)

print(f'Отправили: текстовый файл')
print(f'Статус: {r.status_code}  (ожидаем 400 = Bad Request)')
print(f'Ответ: {json.dumps(r.json(), indent=2, ensure_ascii=False)}')
print('\n✅ Сервис НЕ упал! Вернул понятную ошибку с правильным HTTP-кодом.')

In [ ]:
# Тест 6: Слишком большой файл

big_data = b'x' * (11 * 1024 * 1024)  # 11 МБ
r = httpx.post('http://localhost:8000/predict',
               files={'file': ('big.jpg', big_data, 'image/jpeg')},
               timeout=10)

print(f'Отправили: файл 11 МБ (лимит: 10 МБ)')
print(f'Статус: {r.status_code}  (ожидаем 400)')
print(f'Ответ: {json.dumps(r.json(), indent=2, ensure_ascii=False)}')
print('\n✅ Защита от больших файлов работает!')

In [ ]:
# Останавливаем сервер
!pkill -f 'uvicorn.*main:app' 2>/dev/null || true
print('✅ Сервер остановлен.')

---
## Шаг 10. Типичные ошибки и как их читать

В реальной разработке вы будете видеть ошибки чаще, чем работающий код. Давайте **нарочно сломаем** наш сервис, чтобы научиться читать ошибки.

In [ ]:
# Интерактивный справочник ошибок

errors_accordion = widgets.Accordion(children=[
    widgets.HTML('''
    <h4>❌ FileNotFoundError: No such file or directory: 'cat_dog_model.pkl'</h4>
    <p><b>Когда:</b> Файл модели удалён, перемещён или не был скопирован на сервер.</p>
    <p><b>Как читать:</b></p>
    <ol>
      <li><code>FileNotFoundError</code> — файл не найден</li>
      <li>Путь в сообщении — именно этот файл ищется</li>
      <li><b>Решение:</b> проверить, что модель лежит по нужному пути</li>
    </ol>
    <p><b>Частая причина:</b> модель обучена на одной машине, а сервис запускается на другой.</p>
    '''),
    widgets.HTML('''
    <h4>❌ RuntimeError: Error(s) in loading state_dict for ResNet</h4>
    <p><b>Когда:</b> Модель сохранена в PyTorch 2.0, а загружается в PyTorch 1.9.</p>
    <p><b>Как читать:</b></p>
    <ol>
      <li><code>RuntimeError</code> — ошибка выполнения</li>
      <li><code>Missing key(s)</code> — модель ожидает параметры, которых нет в файле</li>
      <li><b>Решение:</b> обновить PyTorch или пересохранить модель</li>
    </ol>
    <p><b>Правило:</b> всегда указывайте версии в requirements.txt!</p>
    '''),
    widgets.HTML('''
    <h4>❌ 422 Unprocessable Entity — «Не удалось прочитать изображение»</h4>
    <p><b>Когда:</b> Пользователь отправил файл, который не является изображением (PDF, DOCX и т.д.).</p>
    <p><b>Как читать:</b></p>
    <ol>
      <li><code>422</code> — данные корректны по формату, но семантически ошибочны</li>
      <li>Сообщение в <code>detail</code> — конкретная причина</li>
      <li><b>Решение:</b> отправить файл в формате JPEG/PNG/WebP</li>
    </ol>
    <p><b>Без валидации:</b> сервис вернул бы 500 (Internal Server Error) — это неправильно!</p>
    '''),
    widgets.HTML('''
    <h4>❌ ConnectionError — «Connection refused»</h4>
    <p><b>Когда:</b> Сервер не запущен или упал.</p>
    <p><b>Как читать:</b></p>
    <ol>
      <li><code>Connection refused</code> — нечему отвечать на этом порту</li>
      <li>Проверьте: запущен ли сервер? Правильный ли порт?</li>
      <li><b>Решение:</b> запустить сервер или проверить логи</li>
    </ol>
    '''),
])

errors_accordion.set_title(0, '📁 Файл модели не найден')
errors_accordion.set_title(1, '🔧 Несовместимая версия PyTorch')
errors_accordion.set_title(2, '🖼 Файл не является изображением')
errors_accordion.set_title(3, '🔌 Сервер не запущен')

display(errors_accordion)
print('\n👆 Раскройте каждый раздел, чтобы увидеть разбор ошибки')

In [ ]:
# Шпаргалка: HTTP-коды

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

codes = [
    ('200', 'OK', 'Всё хорошо, вот результат', 'lightgreen'),
    ('400', 'Bad Request', 'Клиент отправил ерунду', 'lightyellow'),
    ('404', 'Not Found', 'Такого эндпоинта нет', 'lightyellow'),
    ('422', 'Unprocessable', 'Данные есть, но неправильные', 'lightsalmon'),
    ('500', 'Server Error', 'Баг на стороне сервера', 'salmon'),
]

for i, (code, name, desc, color) in enumerate(codes):
    y = 4.5 - i * 0.9
    ax.add_patch(plt.Rectangle((0.5, y-0.3), 1.5, 0.6, fc=color, ec='gray', lw=1.5))
    ax.text(1.25, y, code, ha='center', va='center', fontsize=14, fontweight='bold')
    ax.text(2.5, y+0.1, name, ha='left', va='center', fontsize=12, fontweight='bold')
    ax.text(2.5, y-0.15, desc, ha='left', va='center', fontsize=10, color='gray')

ax.text(5, 5.2, '4xx = вина клиента  |  5xx = вина сервера', 
        ha='center', fontsize=12, fontstyle='italic', color='gray')

ax.set_xlim(0, 10)
ax.set_ylim(0, 5.5)
ax.set_title('HTTP-коды: шпаргалка', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Шаг 11. Практические задания

Теперь ваша очередь! Каждое задание — это возможность **сломать код, починить и научиться**.

In [ ]:
# Интерактивный список заданий

tasks = widgets.Accordion(children=[
    widgets.HTML('''
    <p>В <code>model_service.py</code> стоит <code>CONFIDENCE_THRESHOLD = 0.7</code>. Попробуйте:</p>
    <ul>
      <li><b>0.3</b> — почти все предсказания «надёжные». Сколько ошибок?</li>
      <li><b>0.95</b> — модель часто не уверена. Сколько отказов?</li>
      <li>Найдите «золотую середину»</li>
    </ul>
    <p><b>Подсказка:</b> протестируйте на 50 картинках и посчитайте: (а) сколько «не уверен», (б) сколько ошибок среди уверенных.</p>
    '''),
    widgets.HTML('''
    <p>Добавьте в <code>main.py</code> эндпоинт <code>POST /predict_batch</code>:</p>
    <pre>@app.post("/predict_batch")
async def predict_batch(files: List[UploadFile] = File(...)):
    results = []
    for file in files:
        # ваш код здесь
    return {"predictions": results}</pre>
    <p><b>Подсказка:</b> не забудьте валидацию для каждого файла!</p>
    '''),
    widgets.HTML('''
    <p>Вот три способа сломать сервис. Для каждого — опишите ошибку и почините:</p>
    <ol>
      <li>Удалите <code>cat_dog_model.pkl</code> и перезапустите сервер</li>
      <li>Замените <code>load_learner(MODEL_PATH)</code> на <code>load_learner("wrong_path.pkl")</code></li>
      <li>Удалите валидацию <code>content_type</code> и отправьте PDF</li>
    </ol>
    '''),
    widgets.HTML('''
    <p>Замените <code>print()</code> на <code>logging</code>:</p>
<pre>import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

logger.info(f"Предсказание: {result}")
logger.error(f"Ошибка: {error}")</pre>
    <p><b>Зачем?</b> В продакшене логи — первые свидетели. Когда что-то сломается в 3 часа ночи, вы будете читать логи.</p>
    '''),
    widgets.HTML('''
    <p>Создайте <code>Dockerfile</code>:</p>
<pre>FROM python:3.10-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]</pre>
    <p><b>Подсказка:</b> файл .pkl тоже нужно COPY. Что будет, если забыть?</p>
    '''),
])

tasks.set_title(0, '🎯 Задание 1: Измените порог уверенности')
tasks.set_title(1, '🎯 Задание 2: Добавьте /predict_batch')
tasks.set_title(2, '🎯 Задание 3: Сломайте сервис и почините')
tasks.set_title(3, '🎯 Задание 4: Добавьте логирование')
tasks.set_title(4, '🎯 Задание 5: Напишите Dockerfile')

display(tasks)

---
## Шаг 12. Идеи для дальнейшего изучения

<img src="https://sfile.chatglm.cn/images-ppt/2d6b62db5b30.png" width="500" align="center">

Вы прошли путь от блокнота до работающего API-сервиса! Вот что дальше:

| Направление | Что изучить |
|-------------|-------------|
| **Деплой** | Docker + Docker Compose, Railway, Render, AWS, GCP |
| **Мониторинг** | Prometheus + Grafana — отслеживание latency, error rate |
| **Версионирование** | MLflow, DVC, Weights & Biases |
| **A/B тестирование** | Запуск двух моделей одновременно |
| **Оптимизация** | ONNX, TensorRT — ускорение инференса |
| **CI/CD** | GitHub Actions — автоматический тест + деплой при каждом коммите |

**Ключевые ресурсы:**
- [FastAPI Documentation](https://fastapi.tiangolo.com/) — одна из лучших документаций в мире Python
- [Full Stack Deep Learning](https://fullstackdeeplearning.com/) — курс про ML в продакшене
- [Made With ML](https://madewithml.com/) — от обучения модели до деплоя
- [3Blue1Brown: Neural Networks](https://www.youtube.com/playlist?list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi) — лучшая визуализация

Вы уже не просто обучаете модели — вы умеете их сервить. Это большой шаг! 🎉